# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")

# Print some main properties
print(f"Dataset identifier: {meta.identifier}\nLicense: {meta.license}\nVersion: {meta.version}\nAuthors: {meta.author}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all RecordSets in the dataset
print('Available RecordSets:')
for record_set in dataset.record_sets:
    print(f"- @id: {record_set.id}, name: {record_set.name}, description: {record_set.description}")

# Preview fields and columns for each RecordSet
print('\nFields and columns for each RecordSet:')
for record_set in dataset.record_sets:
    print(f'\nRecordSet @id: {record_set.id} -- {record_set.name}')
    print('  Fields:')
    for field in record_set.fields:
        colnames = [col.id for col in field.columns] if hasattr(field, 'columns') and field.columns else []
        print(f"    - @id: {field.id}, name: {field.name}, dataType: {field.data_type}, columns: {colnames}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Select the main RecordSet for protein analysis table
# Locate its @id. For this FAIR2 dataset, the main data is typically in the 'Proteins' table.
record_set_ids = [rs.id for rs in dataset.record_sets]
print(f'All RecordSet ids: {record_set_ids}')

# For demonstration, pick the first RecordSet (likely the main table of proteins)
if record_set_ids:
    chosen_record_set = record_set_ids[0]
else:
    raise ValueError('No RecordSets found in dataset!')

# Load records into DataFrame
df = pd.DataFrame(list(dataset.records(record_set=chosen_record_set)))
print(f'Columns in RecordSet {chosen_record_set}:')
print(df.columns.tolist())

# Preview records
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Identify a numeric field (e.g. a protein abundance column or molecular weight)
numeric_candidates = [col for col in df.columns if df[col].dtype in ['float64', 'int64']]
print('Numeric columns:', numeric_candidates)

numeric_field = None
for col in [
    'Molecular Weight (MW)', 'MW', 'Abundance', 'PeptideCount', 'NormalizedAbundance', 'proteinAbundance', 'averageIntensity', 'Score']:
    if col in df.columns:
        numeric_field = col
        break
if numeric_field is None and numeric_candidates:
    numeric_field = numeric_candidates[0]
if numeric_field is None:
    raise ValueError('No numeric field identified! Please review the columns printed above.')

# Filter records for values greater than a threshold
threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with '{numeric_field}' > {threshold:.2f} (mean):")
print(filtered_df.head())

# Normalize the numeric field (z-score)
filtered_df = filtered_df.copy()
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized '{numeric_field}' for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a categorical field, e.g. 'Sample', 'ExperimentGroup', or 'accession'
group_field = None
for candidate in ['Sample', 'ExperimentGroup', 'accession', 'Accession', 'protein']:
    if candidate in df.columns:
        group_field = candidate
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped data by '{group_field}':")
    print(grouped_df.head())
else:
    print('No appropriate group field found in columns.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

# Histogram of the main numeric field
plt.figure(figsize=(8, 5))
df[numeric_field].hist(bins=30, color='skyblue', edgecolor='k')
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# If grouped, show top 10 groups barplot
if group_field:
    plt.figure(figsize=(10, 5))
    top = grouped_df.sort_values(numeric_field, ascending=False).head(10)
    plt.bar(top[group_field], top[numeric_field], color='orange')
    plt.title(f'Top 10 {group_field}s by mean {numeric_field}')
    plt.xlabel(group_field)
    plt.ylabel(f'Mean {numeric_field}')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR² dataset describing mass spectrometry analysis of extracellular vesicles from human mast cells. Using `mlcroissant`, we demonstrated how to:
- Load and inspect dataset metadata via the Croissant schema
- Enumerate record sets, fields, and column IDs
- Load a main data table and perform basic filtering and normalization
- Group and summarize protein metrics by relevant fields
- Visualize feature distributions and group comparisons

This workflow can be adapted to other Croissant datasets for interoperable FAIR machine learning research. Explore further by analyzing fields, linking to experimental metadata, or building ML models.
